# Fetch US Stock Options Data
This notebook fetches option data for US stocks and creates a DataFrame with the following filters:
- Only call options
- Includes delta information
- Within 3 weeks of the closest earnings date
- Earnings on Fridays
- Time of earnings announcement (during market hours or outside)

In [5]:
# Import Required Libraries
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [10]:
# Fetch US Stock Option Data
def fetch_option_data(ticker):
    stock = yf.Ticker(ticker)
    options = stock.option_chain()
    return options.calls

# Example usage
example_ticker = 'AAPL'
options_data = fetch_option_data(example_ticker)
options_data.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency
0,AAPL260506C00215000,2026-05-05 16:47:50+00:00,215.0,67.17,0.0,0.0,0.0,0.0,14.0,8,0.00001,True,REGULAR,USD
1,AAPL260506C00220000,2026-05-05 15:17:00+00:00,220.0,59.95,0.0,0.0,0.0,0.0,8.0,8,0.00001,True,REGULAR,USD
2,AAPL260506C00235000,2026-05-05 19:52:30+00:00,235.0,48.88,0.0,0.0,0.0,0.0,60.0,64,0.00001,True,REGULAR,USD
3,AAPL260506C00242500,2026-05-04 16:11:03+00:00,242.5,33.65,0.0,0.0,0.0,0.0,3.0,4,0.00001,True,REGULAR,USD
4,AAPL260506C00245000,2026-05-05 19:50:27+00:00,245.0,38.79,0.0,0.0,0.0,0.0,1.0,1,0.00001,True,REGULAR,USD


In [13]:
# Display the columns of the calls DataFrame to identify the correct column names
print("Columns in calls DataFrame:", options_data.columns)

Columns in calls DataFrame: Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency'],
      dtype='object')


In [7]:
# Filter for Call Options
# Assuming `options_data` is the DataFrame fetched earlier
call_options = options_data[options_data['optionType'] == 'call']
call_options.head()

KeyError: 'optionType'

In [ ]:
# Filter by Delta Information
# Assuming `call_options` has a 'delta' column
delta_filtered = call_options[(call_options['delta'] > 0.3) & (call_options['delta'] < 0.7)]
delta_filtered.head()

In [ ]:
# Filter by Earnings Date (Within 3 Weeks)
def filter_by_earnings_date(options, earnings_dates):
    today = datetime.now()
    three_weeks = today + timedelta(weeks=3)
    filtered = options[options['earningsDate'].between(today, three_weeks)]
    return filtered

# Example usage
earnings_dates = pd.DataFrame({'earningsDate': ['2026-05-15', '2026-05-22']})
filtered_options = filter_by_earnings_date(delta_filtered, earnings_dates)
filtered_options.head()

In [ ]:
# Filter for Earnings on Fridays
filtered_options['earningsDay'] = pd.to_datetime(filtered_options['earningsDate']).dt.day_name()
friday_options = filtered_options[filtered_options['earningsDay'] == 'Friday']
friday_options.head()

In [ ]:
# Add Time of Earnings Announcement
# Assuming `friday_options` has a 'announcementTime' column
def categorize_announcement_time(row):
    if 'before market' in row['announcementTime'].lower():
        return 'Before Market'
    elif 'after market' in row['announcementTime'].lower():
        return 'After Market'
    else:
        return 'During Market'

friday_options['announcementCategory'] = friday_options.apply(categorize_announcement_time, axis=1)
friday_options.head()

In [14]:
# Import Required Libraries
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
from scipy.stats import norm

# Black-Scholes Model for Greeks Calculation
def calculate_greeks(S, K, T, r, sigma, option_type='call'):
    """
    Calculate option Greeks using the Black-Scholes model.
    S: Current stock price
    K: Strike price
    T: Time to expiration (in years)
    r: Risk-free rate
    sigma: Volatility
    option_type: 'call' or 'put'
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    delta = norm.cdf(d1) if option_type == 'call' else -norm.cdf(-d1)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T)
    theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) -
             r * K * np.exp(-r * T) * norm.cdf(d2 if option_type == 'call' else -d2))
    return delta, gamma, vega, theta

# Fetch Options Chain
def fetch_options_chain(ticker):
    stock = yf.Ticker(ticker)
    options = stock.option_chain()
    calls, puts = options.calls, options.puts
    return calls, puts

# Fetch Historical Prices
def fetch_historical_prices(ticker, start_date, end_date):
    stock = yf.Ticker(ticker)
    return stock.history(start=start_date, end=end_date)

# Fetch Earnings Data
def fetch_earnings_data(ticker):
    stock = yf.Ticker(ticker)
    return stock.calendar

# Combine Data
def combine_data(ticker):
    # Fetch options chain
    calls, puts = fetch_options_chain(ticker)

    # Fetch historical prices
    today = datetime.now()
    start_date = today - timedelta(days=30)
    end_date = today
    historical_prices = fetch_historical_prices(ticker, start_date, end_date)

    # Fetch earnings data
    earnings_data = fetch_earnings_data(ticker)

    # Add Greeks to calls
    expiration_date = datetime.strptime(calls.index[0], '%Y-%m-%d')  # Assuming expiration is in the index
    calls['delta'], calls['gamma'], calls['vega'], calls['theta'] = zip(*calls.apply(
        lambda row: calculate_greeks(
            S=historical_prices['Close'].iloc[-1],  # Latest price
            K=row['strike'],
            T=(expiration_date - today).days / 365,
            r=0.01,  # Assume 1% risk-free rate
            sigma=0.2,  # Assume 20% volatility
            option_type='call'
        ), axis=1
    ))

    # Merge with earnings data
    calls['earningsDate'] = earnings_data.loc['Earnings Date'].values[0]

    # Merge with historical prices
    calls['underlyingPrice'] = historical_prices['Close'].iloc[-1]

    return calls

# Example Usage
ticker = 'AAPL'
combined_data = combine_data(ticker)
print(combined_data.head())

TypeError: strptime() argument 1 must be str, not int